# MediHelper 1.0

In [1]:
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.memory import ConversationBufferMemory
# from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv
from pydantic import Field
from typing import Dict
import pandas as pd
import numpy as np
import re
import os

load_dotenv()

True

In [2]:
df = pd.read_csv('../datasets/medicine_dataset.csv')

In [3]:
df.head(1)

,medicine name,alternative medicines,medicine usages,medicine side effects,habit forming,chemical class,action class,therapeutic class,medicine combined information
0,augmentin 625 duo tablet,"['penciclav 500 mg/125 mg tablet', 'moxikind-c...",['treatment of bacterial infections'],"['vomiting', 'nausea', 'diarrhea']",no,unknown,unknown,anti infectives,medicine name : augmentin 625 duo tablet | alt...


In [4]:
df.shape

(222825, 9)

Environment Variables

In [5]:
gemini_api_key = os.getenv('GEMINI_API_KEY')
langchain_api_key = os.getenv('LANGCHAIN_API_KEY')

if gemini_api_key is None:
    raise ValueError("GEMINI_API_KEY environment variable is not set.")

if langchain_api_key is None:
    raise ValueError("LANGCHAIN_API_KEY environment variable is not set.")

os.environ['GOOGLE_API_KEY'] = gemini_api_key

Initialization

In [6]:
# llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash', temperature=0)
llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0)
embedding = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
persist_directory = '../chromadb'

### Creating Embeddings

In [ ]:
from tqdm import tqdm
import math

batch_size = 1000
num_rows = len(df)
num_batches = math.ceil(num_rows / batch_size)

# Initialize Chroma only once
vectordb = None

for i in tqdm(range(num_batches)):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, num_rows)
    
    texts_batch = df['medicine combined information'][start_idx:end_idx].tolist()
    
    if i == 0:
        vectordb = Chroma.from_texts(
            texts=texts_batch,
            embedding=embedding,
            persist_directory=persist_directory
        )
    else:
        vectordb.add_texts(texts_batch)

print(vectordb._collection.count())

In [9]:
print(vectordb._collection.count())

222825


### Results

Getting the data from vector db

In [ ]:
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

print(vectordb._collection.count())

Prompt Templating

In [11]:
template = """
You are a knowledgeable and helpful pharmacist assistant with expertise in medicines, diseases, side-effects, usages, alternative medicines, and various classes of medicines such as action class, therapeutic class, and chemical class. Use the following pieces of context provided by the doctor to answer the question at the end. The context contains information in key-value pairs and each piece of information is separated by a pipe symbol (|). Use only the relevant information from the context to answer the question.

Important:
1. Provide accurate and concise answers based on the context.
2. If the context does not provide sufficient information to answer the question, state, "I don't know based on the provided context."
3. Do not fabricate or assume information that is not explicitly provided in the context.
4. Give the answer in a properly formatted manner

Context:
{context}

Question: {question}
Answer:
"""
prompt = PromptTemplate.from_template(template)

Formatting Functions

In [12]:
def format_dialogue(chat_history):
    formatted_dialogue = "Chat History:\n\n"
    history_str = chat_history['history']
    
    # Use regular expression to split based on 'AI:' or 'Human:'
    history_parts = re.split(r'\n+(?=AI:|Human:)', history_str)
    
    # Use a set to track unique dialogue lines
    unique_dialogues = set()
    
    for part in history_parts:
        if part.startswith('AI:'):
            speaker = 'AI'
        elif part.startswith('Human:'):
            speaker = 'Human'
        else:
            continue  # Skip any unmatched parts
        
        dialogue = f"{speaker}: {part.strip()}"
        
        # Check if dialogue is unique before appending
        if dialogue not in unique_dialogues:
            formatted_dialogue += f"{dialogue}\n\n"
            unique_dialogues.add(dialogue)
    
    return formatted_dialogue.strip()


def format_context(context_str):
    formatted_context = "More Context:\n\n"
    formatted_context += context_str
    return formatted_context

Adding a limit to the chat history memory

In [13]:
class LimitedConversationBufferMemory(ConversationBufferMemory):
    max_limit: int = Field(default=20, alias="max_limit")

    def __init__(self, memory_key="history", input_key="question", output_key="response", max_limit=20):
        super().__init__(memory_key=memory_key, input_key=input_key, output_key=output_key)
        self.max_limit = max_limit

    def is_memory_full(self) -> bool:
        """
        Check if memory is full based on the max_limit.
        """
        return len(self.chat_memory.messages) >= self.max_limit
    
    def delete_oldest_item(self):
        """
        Delete the oldest item from memory.
        """
        if self.chat_memory.messages:
            self.chat_memory.messages.pop(0)
    
    def save_context_with_limit_check(self, inputs: Dict[str, str], outputs: Dict[str, str]):
        """
        Save context to memory after checking and managing memory limits.
        """
        if self.is_memory_full():
            self.delete_oldest_item()
        super().save_context(inputs, outputs)


In [14]:
# Initialize the custom memory with a limit of 5 interactions
memory = LimitedConversationBufferMemory(max_limit=10)

Chat without history

In [15]:
def chat_without_history(question):
    retriever=vectordb.as_retriever(
        # search_type="mmr",
        search_kwargs={"k": 5}
    )   
    context_str = retriever.invoke(question)
    response=llm.invoke(prompt.format(context=context_str , question=question))
    return response.content

Chat with history

In [16]:
# Function to ask a question with memory context
def chat_with_history(question):
    retriever = vectordb.as_retriever(
        search_kwargs={"k": 5}
    )
    context_docs = retriever.invoke(question)
    context_str = "\n\n".join([doc.page_content for doc in context_docs])
    formatted_context_str = format_context(context_str)
    
    # Load previous chat history from memory
    chat_history = memory.load_memory_variables({})
    chat_history_str = format_dialogue(chat_history)
    
    # Combine chat history and new context
    combined_context = chat_history_str + "\n\n" + formatted_context_str
    
    # Generate the final response considering both chat history and new context
    final_response = llm.invoke(prompt.format(context=combined_context, question=question))
    
    # Save the final response to memory if it's a unique dialogue line
    if final_response.content:
        dialogue_to_save = f"AI: {final_response.content.strip()}"
        if dialogue_to_save not in chat_history['history']:
            memory.save_context_with_limit_check({"question": question}, {"response": final_response.content})
    
    return final_response.content

Questions

In [17]:
question1 = "What are the usages for azithral 500 tablet?"
question2 = "What are the side effects of azithral 500 tablet?"
question3 = "What are the alternative medicines of azithral 500 tablet?"
question4 = "Are the suggested medicines habit forming?"
question5 = "What are the chemical class of the suggested medicines?"
question6 = "What are the action class of the suggested medicines?"
question7 = "What are the therapeutic class of the suggested medicines?"
question8 = "What are the medicines used for treating bacterial infections?"
question9 = 'What are the side effects of Moxiforce-cv 625 tablet'
question10 = 'What are the classification of Baciclox plus capsule'
question11 = 'What are the alternative medicines of azithral 500 tablet?'
question12 = 'What are the medicines used to treat bacterial infection?'
question13 = 'What are the medicines used to treat bacterial infection and also give details about the suggested medicines?'

Printing the Results

In [18]:
result= chat_without_history(question1)
if result:
    print(result)
else:
    print(f"No answer found")

The usages for azithral 500 tablet are: treatment of bacterial infections.


In [19]:
result= chat_without_history(question2)
if result:
    print(result)
else:
    print(f"No answer found")

The side effects of azithral 500 tablet are vomiting, nausea, abdominal pain, and diarrhea.


In [20]:
result= chat_without_history(question12)
if result:
    print(result)
else:
    print(f"No answer found")

The medicines used to treat bacterial infections are: brodcillin 250mg/250mg tablet, mybactim 500mg/500mg injection, briclox 250mg/250mg tablet, bicil 250 mg/250 mg tablet, and bicilin d 250 mg/250 mg tablet.


In [ ]:
result= chat_without_history(question13)
if result:
    print(result)
else:
    print(f"No answer found")

In [22]:
result = chat_with_history(question1)
if result:
    print(result)
else:
    print(f"No answer found")

The usages for azithral 500 tablet are: treatment of bacterial infections.


In [23]:
result = chat_with_history(question2)
if result:
    print(result)
else:
    print(f"No answer found")

The side effects of azithral 500 tablet are vomiting, nausea, abdominal pain, and diarrhea.


In [24]:
result = chat_with_history(question3)
if result:
    print(result)
else:
    print(f"No answer found")

The alternative medicines for azithral 500 tablet are zithrocare 500mg tablet, azax 500 tablet, zady 500 tablet, cazithro 500mg tablet, and trulimax 500mg tablet.


In [25]:
chat_history = memory.load_memory_variables({})
chat_history_str = format_dialogue(chat_history)
print(chat_history_str)

Chat History:

Human: Human: What are the usages for azithral 500 tablet?

AI: AI: The usages for azithral 500 tablet are: treatment of bacterial infections.

Human: Human: What are the side effects of azithral 500 tablet?

AI: AI: The side effects of azithral 500 tablet are vomiting, nausea, abdominal pain, and diarrhea.

Human: Human: What are the alternative medicines of azithral 500 tablet?

AI: AI: The alternative medicines for azithral 500 tablet are zithrocare 500mg tablet, azax 500 tablet, zady 500 tablet, cazithro 500mg tablet, and trulimax 500mg tablet.
